# 04 · DCC-GARCH: Time-Varying Correlation
**Brazilian Stock-Bond Correlation Study**

Uses Engle (2002) two-stage DCC-GARCH to estimate the *daily* evolution of the
Ibovespa–bond correlation over 20 years.

1. Stage 1: univariate GARCH(1,1) per asset via `arch`
2. Stage 2: DCC parameters (a, b) via MLE on standardised residuals
3. Time-varying ρ_t chart — the academic complement to notebook 03's rolling window
4. DCC vs EMBI scatter: does sovereign risk drive correlation?
5. Crisis regime averages and persistence analysis

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from arch import arch_model
from scipy.optimize import minimize

from fetch import load_master, CRISES, REGIMES

master = load_master()

plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
})
CRISIS_COLORS = {
    "GFC":"#d62728","Dilma":"#ff7f0e","Joesley":"#9467bd",
    "COVID":"#2ca02c","Americanas":"#8c564b","Fiscal24":"#e377c2",
}
LABELS = {"ibov":"Ibovespa","ntnb":"NTN-B 5y","ltn":"LTN 2y",
          "ntnf":"NTN-F 10y","lft":"LFT 1y"}

def add_crisis_bands(ax, alpha=0.15):
    for name, (s, e) in CRISES.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e),
                   color=CRISIS_COLORS[name], alpha=alpha, label=name)

## 1. Stage 1: Fit univariate GARCH(1,1) per asset

In [ ]:
RET_COLS = ["ibov", "ntnb", "ltn", "ntnf", "lft"]

def fit_garch11(series, name=""):
    """Fit GARCH(1,1) and return standardised residuals + conditional volatility."""
    am  = arch_model(series, vol='GARCH', p=1, q=1, dist='normal', rescale=False)
    res = am.fit(disp='off', show_warning=False)
    std_resid = res.resid / res.conditional_volatility
    params = {k: float(v) for k, v in res.params.items()}
    print(f"  {name:<18}  omega={params.get('omega',0):.5f}  "
          f"alpha[1]={params.get('alpha[1]',0):.4f}  "
          f"beta[1]={params.get('beta[1]',0):.4f}  "
          f"persist={params.get('alpha[1]',0)+params.get('beta[1]',0):.4f}")
    return std_resid, res.conditional_volatility

print("=== GARCH(1,1) parameters (scaled returns × 100) ===")
std_resids = {}
cond_vols  = {}
for col in RET_COLS:
    s = master[col].dropna() * 100
    sr, cv = fit_garch11(s, LABELS[col])
    std_resids[col] = sr
    cond_vols[col]  = cv

In [ ]:
# Plot conditional volatility for Ibovespa and NTN-B
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

for ax, col, color in zip(axes, ["ibov","ntnb"], ["#1f77b4","#d62728"]):
    cv = cond_vols[col]
    ax.fill_between(cv.index, cv * np.sqrt(252),
                    color=color, alpha=0.5, label=f"{LABELS[col]} ann. vol")
    add_crisis_bands(ax, alpha=0.12)
    ax.set_ylabel("Annualised volatility (%)")
    ax.set_title(f"GARCH(1,1) conditional volatility — {LABELS[col]}", fontsize=11)
    ax.legend(fontsize=9)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
plt.savefig("../outputs/fig_garch_volatility.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_garch_volatility.png")

## 2. Stage 2: DCC parameter estimation

Estimate the DCC parameters (a, b) by maximum likelihood on the standardised
residuals, **with standard errors from the numerical Hessian**.

`a` governs how strongly the conditional correlation responds to news. Reporting
`a` without a standard error makes "correlations are time-varying" an assertion;
with one it is a testable hypothesis (H0: a = 0). The estimator lives in
`src/metrics.py::fit_dcc` and is unit-tested against simulated DCC processes with
known parameters, including the degenerate constant-correlation case.

In [ ]:
from metrics import fit_dcc

print("=== DCC-GARCH(1,1) parameter estimates ===")
print(f"  {'pair':<28} {'a':>8} {'SE(a)':>8} {'t(a)':>7} {'b':>8} "
      f"{'persist':>8} {'mean rho':>9} {'sd rho':>7}")

dcc_results = {}
pairs = [("ibov","ntnb"), ("ibov","ltn"), ("ibov","ntnf"), ("ibov","lft")]
for ca, cb in pairs:
    f = fit_dcc(std_resids[ca], std_resids[cb])
    dcc_results[(ca, cb)] = f
    flag = ("  <- not identified" if not f["identified"]
            else "  <- a>0 at 5%" if f["t_a"] > 1.96 else "")
    print(f"  {'Ibovespa x ' + LABELS[cb]:<28} {f['a']:>8.4f} {f['se_a']:>8.4f} "
          f"{f['t_a']:>7.2f} {f['b']:>8.4f} {f['persistence']:>8.4f} "
          f"{f['rho'].mean():>9.3f} {f['rho'].std():>7.3f}{flag}")

print("\nH0: a = 0 (constant conditional correlation). |t| > 1.96 rejects at 5%.")
print("A pair that fails to reject is NOT evidence of crisis correlation spikes,")
print("however suggestive the plotted path looks.")

## 3. The DCC correlation chart — Figure 6

Time-varying correlation ρ_t from DCC-GARCH. This is the **formal econometric**
complement to the rolling window chart in notebook 03.

In [ ]:
bond_cols  = ["ntnb", "ltn", "ntnf", "lft"]
colors     = ["#d62728","#ff7f0e","#2ca02c","#9467bd"]

fig, ax = plt.subplots(figsize=(14, 5.5))

for col, color in zip(bond_cols, colors):
    rho = dcc_results[("ibov", col)]["rho"]
    ax.plot(rho.index, rho, label=LABELS[col], lw=1.4, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=1.2, ls="--", alpha=0.7, label="ρ = 0")
add_crisis_bands(ax, alpha=0.13)
ax.axvline(pd.Timestamp("2020-01-01"), color="navy",
           lw=1.5, ls=":", alpha=0.8, label="IMF DM regime shift")

ax.set_ylim(-0.3, 0.5)
ax.set_ylabel("DCC-GARCH daily conditional correlation ρ_t", fontsize=11)
ax.set_title(
    "DCC-GARCH(1,1): Ibovespa vs. Brazilian bond indices — daily conditional correlation\n"
    "Brazil, 2004–2026  (Engle 2002)",
    fontsize=13,
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

asset_handles  = [plt.Line2D([0],[0], color=c, lw=2, label=LABELS[col])
                  for col,c in zip(bond_cols, colors)]
asset_handles += [plt.Line2D([0],[0], color="black", ls="--", lw=1.5, label="ρ=0"),
                  plt.Line2D([0],[0], color="navy",  ls=":",  lw=1.5, label="IMF DM shift")]
crisis_handles = [plt.Rectangle((0,0),1,1, fc=CRISIS_COLORS[n], alpha=0.4, label=n)
                  for n in CRISES]
ax.legend(handles=asset_handles + crisis_handles,
          loc="lower left", fontsize=8, ncol=3)

plt.tight_layout()
plt.savefig("../outputs/fig_dcc_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_dcc_correlation.png")

## 4. DCC correlation vs. EMBI sovereign risk

In [ ]:
rho_ntnb = dcc_results[("ibov","ntnb")]["rho"]
embi_aligned = master["embi"].reindex(rho_ntnb.index).ffill()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Time series overlay
ax = axes[0]
ax2 = ax.twinx()
ax.plot(rho_ntnb.index, rho_ntnb, color="#d62728", lw=1.3, label="DCC ρ_t (left)")
ax2.plot(embi_aligned.index, embi_aligned, color="#1f77b4",
         lw=1, alpha=0.6, label="EMBI % (right)")
add_crisis_bands(ax, alpha=0.1)
ax.set_ylabel("DCC ρ_t (Ibovespa × NTN-B)", color="#d62728", fontsize=10)
ax2.set_ylabel("EMBI+ Brazil (%)", color="#1f77b4", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title("DCC correlation vs. EMBI sovereign risk", fontsize=11)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

# Scatter
ax3 = axes[1]
df_scatter = pd.DataFrame({"rho": rho_ntnb, "embi": embi_aligned}).dropna()
crisis_label = master["crisis"].reindex(df_scatter.index).fillna("None")
for cname, group in df_scatter.groupby(crisis_label):
    color = CRISIS_COLORS.get(cname, "#aaaaaa")
    alpha = 0.7 if cname != "None" else 0.15
    size  = 12  if cname != "None" else 3
    ax3.scatter(group["embi"], group["rho"], s=size,
                color=color, alpha=alpha,
                label=cname if cname != "None" else None)

# OLS trend line
from numpy.polynomial import polynomial as P
x = df_scatter["embi"].values
y = df_scatter["rho"].values
coeffs = np.polyfit(x, y, 1)
xline  = np.linspace(x.min(), x.max(), 100)
ax3.plot(xline, np.polyval(coeffs, xline), "k--", lw=1.5)
r2 = np.corrcoef(x, y)[0,1]**2
ax3.set_xlabel("EMBI+ Brazil (%)", fontsize=10)
ax3.set_ylabel("DCC ρ_t", fontsize=10)
ax3.set_title(f"Scatter: DCC ρ vs. EMBI  (R²={r2:.3f})", fontsize=11)
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/fig_dcc_vs_embi.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"EMBI → DCC correlation R² = {r2:.3f}")
print("(Higher R² = sovereign risk is primary driver of stock-bond correlation)")

## 5. Crisis-period DCC correlation summary table

In [ ]:
rows = []
for cname, (s, e) in list(CRISES.items()) + [("Full sample", ("2004-01-01","2026-12-31"))]:
    row = {"Period": cname}
    for ca, cb in [("ibov","ntnb"),("ibov","ltn"),("ibov","ntnf"),("ibov","lft")]:
        rho = dcc_results[(ca,cb)]["rho"]
        mask = (rho.index >= s) & (rho.index <= e)
        row[f"Ibov x {LABELS[cb]}"] = round(rho[mask].mean(), 3) if mask.sum() else np.nan
    rows.append(row)

dcc_tbl = pd.DataFrame(rows).set_index("Period")
print("=== DCC-GARCH average conditional correlation by period ===")
print(dcc_tbl.to_string())
dcc_tbl.to_csv("../outputs/nb_tbl_dcc_crisis_correlations.csv")

# A crisis average above the full-sample average is only suggestive: the DCC path is
# itself estimated from returns whose variance explodes in a crisis. Notebook 03's
# Forbes-Rigobon table is the test of whether that elevation survives the volatility
# adjustment.
fs = dcc_tbl.loc["Full sample"]
print("\n=== Crisis elevation relative to the full sample ===")
for cname in CRISES:
    r = dcc_tbl.loc[cname]
    parts = "  ".join(f"{c.split(' x ')[1]}: {r[c]/fs[c]:.2f}x" for c in dcc_tbl.columns
                      if np.isfinite(r[c]) and abs(fs[c]) > 1e-6)
    print(f"  {cname:12s} {parts}")
print("\nSaved: outputs/nb_tbl_dcc_crisis_correlations.csv")

## ✅ Notebook 04 complete

**What to read off the output above** (values are computed, not asserted here):

| Question | Where to look |
|----------|---------------|
| Are correlations genuinely time-varying? | `t(a)` in the stage-2 table. \|t\| > 1.96 rejects a = 0. |
| How persistent? | `persist` = a + b. Near 1 means shocks to correlation decay slowly. |
| Do correlations spike in crises? | The elevation table — but see the caveat below. |
| Is the LFT pair meaningful? | It is flagged *not identified*: the LFT return series is
  near-deterministic, so its conditional correlation path is degenerate by construction
  and its DCC row should not be interpreted. |

**Caveat on crisis spikes.** A DCC path that rises in a crisis is not by itself evidence
that the propagation mechanism strengthened. The conditional correlation is estimated
from returns whose variance rises several-fold in the same window, and Forbes & Rigobon
(2002) show that this alone biases measured correlation upward. Notebook 03 reports the
volatility-adjusted comparison; treat that as the test and this table as the description.

**Next:** `05_copula.ipynb` — tail dependence coefficients